# Actividad 4: Aplicación de algoritmos de aprendizaje no supervisado con PySpark

**Materia:** Análisis de grandes volúmenes de datos  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Autor:** Jonathan Javier Monsalve Giraldo (A01840272)  
**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 1 de junio de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025  
**Modalidad:** Individual


## Objetivo

Aplicar algoritmos de aprendizaje no supervisado en PySpark MLlib sobre una muestra M' derivada de la muestra estratificada M construida en la Etapa 2 del proyecto del equipo. El problema elegido es la segmentación de viajes Yellow Taxi NYC en arquetipos operativos, sin utilizar una variable objetivo, para descubrir perfiles naturales de viaje según distancia, duración, velocidad, zona de origen, horario, pasajeros y régimen Flex Fare.

## Estructura del notebook

1. **Introducción**: aprendizaje no supervisado, algoritmos representativos y los disponibles en PySpark MLlib.
2. **Selección de los datos**: reconstrucción compacta de M, construcción de la muestra individual M' y validación de representatividad.
3. **Preparación del conjunto de entrenamiento y prueba**: partición estratificada train/test y validación del split como revisión de estabilidad.
4. **Construcción de modelos de aprendizaje no supervisado**: selección de features, pipeline, barrido de k, KMeans, comparación con GMM, evaluación e interpretación de arquetipos.

### Nota para el profesor

Las secciones 2.0, 2.1 y 2.2 reutilizan de forma compacta la reconstrucción de M desde la Etapa 2, la construcción de M' y la validación de representatividad usadas en la Actividad 3. Se conservan para que el notebook sea autocontenido y reproducible. El aporte nuevo de esta actividad inicia en la **Sección 3**, donde el split se usa para medir estabilidad del patrón, y sobre todo en la **Sección 4**, donde se derivan las variables operativas para clustering y se entrenan e interpretan los modelos no supervisados.


## 1. Introducción

### 1.1 Aprendizaje no supervisado

El aprendizaje no supervisado agrupa métodos que buscan estructura interna en datos sin una variable objetivo conocida. A diferencia del aprendizaje supervisado, donde cada observación incluye una etiqueta para entrenar y evaluar predicciones, aquí el modelo identifica similitudes, patrones, componentes latentes o casos poco usuales a partir de las variables disponibles.

La calidad de un resultado no supervisado no se mide comparando contra una respuesta verdadera, sino con criterios internos y con interpretación del dominio. Por ello, métricas como cohesión, separación, varianza explicada, probabilidad de pertenencia o frecuencia de patrones se usan como guías, pero deben complementarse con una lectura razonada de los grupos o estructuras descubiertas.

### 1.2 Familias representativas

Las técnicas no supervisadas se organizan en varias familias. El **clustering** agrupa observaciones similares; KMeans representa cada grupo por un centroide, BisectingKMeans construye divisiones jerárquicas y Gaussian Mixture Model permite pertenencia probabilística a componentes gaussianos. La **reducción de dimensión**, como PCA o SVD, resume muchas variables en menos componentes que conservan variabilidad. Las **reglas de asociación**, como FPGrowth, descubren combinaciones frecuentes de ítems o eventos. En **texto**, LDA identifica temas latentes a partir de vectores de conteos. La **detección de anomalías** puede aproximarse con distancia al centroide, baja probabilidad de pertenencia o reglas de negocio cuando no existe una etiqueta de fraude o error.

### 1.3 Algoritmos disponibles en PySpark MLlib

PySpark expone estas técnicas mediante la API moderna `pyspark.ml`, basada en DataFrames y en el patrón `Estimator`/`Transformer`/`Pipeline`. Un `Estimator` aprende parámetros con `fit()`, un `Transformer` agrega columnas con `transform()`, y un `Pipeline` encadena preparación de datos y modelo para aplicar en test las transformaciones aprendidas en train.

| Familia | Submódulo PySpark | Implementaciones relevantes | Uso típico |
|---|---|---|---|
| Clustering | `pyspark.ml.clustering` | `KMeans`, `BisectingKMeans`, `GaussianMixture`, `LDA`, `PowerIterationClustering` | Segmentación tabular, temas en texto o clusters sobre grafos |
| Reducción de dimensión | `pyspark.ml.feature` | `PCA` | Compresión de variables, visualización o preprocesamiento antes de clustering |
| Patrones frecuentes | `pyspark.ml.fpm` | `FPGrowth`, `PrefixSpan` | Canastas, secuencias y combinaciones frecuentes de eventos |
| Evaluación | `pyspark.ml.evaluation` | `ClusteringEvaluator` | Cálculo de silhouette como métrica interna de cohesión y separación |

La tabla resume las clases de alto nivel disponibles en la API DataFrame `pyspark.ml`, no las clases auxiliares de modelo, resumen o la API RDD antigua `pyspark.mllib`. La documentación oficial de Spark lista en clustering `KMeans`, `LDA`, `BisectingKMeans`, `GaussianMixture` y `PowerIterationClustering`; en patrones frecuentes, `FPGrowth` y `PrefixSpan`; y en transformación de features, `PCA` para reducción de dimensión. KMeans, BisectingKMeans y GMM producen una columna de cluster y pueden evaluarse con `ClusteringEvaluator` mediante silhouette. En no supervisado esta métrica no equivale a una verdad externa: se usa como criterio interno junto con tamaño de clusters, estabilidad train/test e interpretación de los perfiles.

### 1.4 Referencias

Apache Software Foundation. (2026). *Clustering - Spark 4.1.2 Documentation*. https://spark.apache.org/docs/latest/ml-clustering.html

Apache Software Foundation. (2026). *Extracting, transforming and selecting features - Spark 4.1.2 Documentation*. https://spark.apache.org/docs/latest/ml-features.html

Apache Software Foundation. (2026). *Frequent Pattern Mining - Spark 4.1.2 Documentation*. https://spark.apache.org/docs/latest/ml-frequent-pattern-mining.html

Polak, A. (2023). *Scaling machine learning with Spark: Distributed ML with MLlib, TensorFlow, and PyTorch*. O'Reilly Media. Capítulo 6, "Training Models with Spark MLlib".


## 2. Selección de los datos

### 2.0 Reconstrucción compacta de la muestra M (recap de Etapa 2)

> **Nota al profesor:** las secciones 2.0, 2.1 y 2.2 reutilizan de forma compacta la reconstrucción de la muestra M desde la Etapa 2, la construcción de M' y la validación de representatividad utilizadas en la Actividad 3. Se conservan para que este notebook sea autocontenido y reproducible. Si ya revisó esa parte en la entrega anterior, puede saltar la lectura detallada de estas secciones y continuar en la Sección 3, donde se reencuadra el split para aprendizaje no supervisado, y en la Sección 4, donde inicia el modelado de clustering.

La muestra M reproduce la construida con el equipo en la Etapa 2. Cada etapa se conserva por una razón metodológica específica:

- **Carga y downcast:** se unifican los 24 parquets mensuales de 2024-2025 y se reducen tipos numéricos para bajar el uso de memoria sin cambiar la semántica de los campos.
- **Filtros destructivos:** se acotan los viajes a rangos físicamente válidos (`trip_distance` 0-200 mi, `fare_amount` 0-1000 USD, `total_amount` 0-1200 USD) y se eliminan registros incoherentes, como distancia cero con tarifa positiva. Esto descarta alrededor de 6% de filas corruptas o atípicas extremas que sesgarían el muestreo y el modelado.
- **Imputaciones documentadas:** se normalizan campos que, nulos o fuera de rango, romperían la construcción del estrato o el pipeline posterior; por ejemplo, `passenger_count` fuera de [1, 6] se lleva a 1, recargos nulos a 0 y `RatecodeID` nulo a 99.
- **Estrato compuesto:** `stratum_id` combina zona de origen, forma de pago, franja horaria y rango de distancia. Estratificar asegura que la muestra preserve la heterogeneidad de la población, no solo los perfiles dominantes.
- **Extracción de M:** `sampleBy` aplica muestreo proporcional por estrato y reduce cerca de 84M viajes limpios a ~5M, con piso mínimo de 500 filas por estrato para conservar perfiles raros.


In [1]:
# Dependencias de Python para el notebook. Idempotente.
!pip install -q pyspark findspark pandas matplotlib

# Solo en Google Colab: descomentar para instalar Java (JVM de Spark).
# Localmente esta línea no es necesaria si Java ya está instalado.
# !apt-get install openjdk-8-jdk-headless -qq > /dev/null


In [2]:
# Setup, descarga idempotente, lectura, downcast, filtros e imputaciones.
import findspark
findspark.init()

from pathlib import Path
import urllib.request

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.debug.maxToStringFields", 100)
    .getOrCreate())

print(f"Spark {spark.version}")

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
YEARS = (2024, 2025)

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(url, target):
    if target.exists():
        return "skip"
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, target)
    return "ok"

for y in YEARS:
    for m in range(1, 13):
        filename = f"yellow_tripdata_{y}-{m:02d}.parquet"
        status = fetch(f"{CDN_BASE}/trip-data/{filename}", DATA_DIR / filename)
        print(f"{status:>6}  {filename}")

status = fetch(f"{CDN_BASE}/misc/taxi_zone_lookup.csv", DATA_DIR / "taxi_zone_lookup.csv")
print(f"{status:>6}  taxi_zone_lookup.csv")

df_native = (spark.read.option("mergeSchema", "true")
    .parquet(*sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))))

zones = (spark.read.option("header", True).option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv")))

df_raw = df_native.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

df_filtered = (df_raw
    .filter(F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    .filter(F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
    .filter(F.col("trip_distance").between(0, 200))
    .filter(F.col("fare_amount").between(0, 1000))
    .filter(F.col("total_amount").between(0, 1200))
    .filter(~((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0))))

n_raw, n_filtered = df_raw.count(), df_filtered.count()
pct_removed = (n_raw - n_filtered) / n_raw * 100
print(f"Crudo: {n_raw:,} | Tras filtros: {n_filtered:,} | Pérdida: {pct_removed:.2f}%")
assert pct_removed < 15.0, f"Filtros removieron {pct_removed:.2f}% > 15%. Revisar datos o filtros."

df_clean = (df_filtered
    .withColumn("passenger_count",
        F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))
         .otherwise(F.lit(1).cast("byte")))
    .withColumn("cbd_congestion_fee",
        F.when(F.col("cbd_congestion_fee").isNull() | (F.col("tpep_pickup_datetime") < F.lit("2025-01-05")),
               F.lit(0.0).cast("float"))
         .otherwise(F.col("cbd_congestion_fee")))
    .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge"), F.lit(0.0).cast("float")))
    .withColumn("Airport_fee", F.coalesce(F.col("Airport_fee"), F.lit(0.0).cast("float")))
    .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), F.lit(99).cast("byte")))
    .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), F.lit("F"))))

imputed_cols = ["passenger_count", "cbd_congestion_fee", "congestion_surcharge",
                "Airport_fee", "RatecodeID", "store_and_fwd_flag"]
nulls = df_clean.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in imputed_cols]).first()
assert all((nulls[c] or 0) == 0 for c in imputed_cols), f"Nulos remanentes: {nulls.asDict()}"
print("Imputaciones aplicadas; 0 nulos en las 6 columnas objetivo.")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/01 01:23:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1
  skip  yellow_tripdata_2024-01.parquet
  skip  yellow_tripdata_2024-02.parquet
  skip  yellow_tripdata_2024-03.parquet
  skip  yellow_tripdata_2024-04.parquet
  skip  yellow_tripdata_2024-05.parquet
  skip  yellow_tripdata_2024-06.parquet
  skip  yellow_tripdata_2024-07.parquet
  skip  yellow_tripdata_2024-08.parquet
  skip  yellow_tripdata_2024-09.parquet
  skip  yellow_tripdata_2024-10.parquet
  skip  yellow_tripdata_2024-11.parquet
  skip  yellow_tripdata_2024-12.parquet
  skip  yellow_tripdata_2025-01.parquet
  skip  yellow_tripdata_2025-02.parquet
  skip  yellow_tripdata_2025-03.parquet
  skip  yellow_tripdata_2025-04.parquet
  skip  yellow_tripdata_2025-05.parquet
  skip  yellow_tripdata_2025-06.parquet
  skip  yellow_tripdata_2025-07.parquet
  skip  yellow_tripdata_2025-08.parquet
  skip  yellow_tripdata_2025-09.parquet
  skip  yellow_tripdata_2025-10.parquet
  skip  yellow_tripdata_2025-11.parquet
  skip  yellow_tripdata_2025-12.parquet
  skip  taxi_zone_lookup.csv

Crudo: 89,892,322 | Tras filtros: 84,437,138 | Pérdida: 6.07%


Imputaciones aplicadas; 0 nulos en las 6 columnas objetivo.


In [3]:
# Construcción del estrato y recálculo del diccionario de fracciones de M.
airport_ids = {r.LocationID for r in zones.filter(F.col("service_zone").isin("Airports", "EWR")).collect()}
unknown_ids = {264, 265}
manhattan_ids = {r.LocationID for r in zones.filter(F.col("Borough") == "Manhattan").collect()} - airport_ids - unknown_ids
outer_ids = {r.LocationID for r in zones.filter(F.col("Borough").isin("Brooklyn", "Queens", "Bronx", "Staten Island")).collect()} - airport_ids - unknown_ids

df_feat = (df_clean
    .withColumn("pu_macro_zone",
        F.when(F.col("PULocationID").isin(sorted(airport_ids)), "airport")
         .when(F.col("PULocationID").isin(sorted(unknown_ids)), "unknown")
         .when(F.col("PULocationID").isin(sorted(manhattan_ids)), "manhattan")
         .when(F.col("PULocationID").isin(sorted(outer_ids)), "outer_borough")
         .otherwise("unknown"))
    .withColumn("payment_group",
        F.when(F.col("payment_type") == 0, "flex")
         .when(F.col("payment_type") == 1, "credit")
         .when(F.col("payment_type") == 2, "cash")
         .otherwise("other"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("day_hour_bucket",
        F.when(F.col("pickup_hour").between(0, 5), "late_night")
         .when(F.col("dow").isin(1, 7), "weekend")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(6, 10), "weekday_am")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(16, 20), "weekday_pm_peak")
         .otherwise("other"))
    .withColumn("trip_distance_bin",
        F.when(F.col("trip_distance") < 1.12, "short")
         .when(F.col("trip_distance") < 12.43, "medium")
         .otherwise("long"))
    .withColumn("is_flex_fare", F.col("payment_type") == 0)
    .withColumn("cbd_period_flag",
        F.when(F.col("tpep_pickup_datetime") < F.lit("2025-01-05"), "pre_cbd").otherwise("post_cbd"))
    .withColumn("stratum_id",
        F.concat_ws("|",
            F.col("pu_macro_zone"), F.col("payment_group"),
            F.col("day_hour_bucket"), F.col("trip_distance_bin"))))

ESTIMATED_M = 5_030_141
N_M_TARGET = 5_000_000
MIN_FLOOR_M = 500

strata_D = (df_feat.groupBy("stratum_id").count()
    .withColumnRenamed("count", "n_D")
    .withColumn("target_n",
        F.least(F.col("n_D"),
                F.greatest(F.lit(MIN_FLOOR_M).cast("long"),
                           F.round(F.lit(N_M_TARGET) * F.col("n_D") / F.lit(n_filtered)).cast("long"))))
    .withColumn("fraction", F.col("target_n") / F.col("n_D")))

fractions = {r["stratum_id"]: float(r["fraction"]) for r in strata_D.select("stratum_id", "fraction").collect()}
assert len(fractions) == 240, f"Se esperaban 240 estratos, se obtuvieron {len(fractions)}"
assert all(0 < f <= 1.0 for f in fractions.values())

print(f"Estratos en D: {len(fractions)}")
print("Variables de estrato construidas:", sorted(set(df_feat.columns) - set(df_clean.columns)))


Estratos en D: 240
Variables de estrato construidas: ['cbd_period_flag', 'day_hour_bucket', 'dow', 'is_flex_fare', 'payment_group', 'pickup_hour', 'pu_macro_zone', 'stratum_id', 'trip_distance_bin']


In [4]:
# Extracción de M mediante sampleBy, con el diccionario de fracciones de Etapa 2.
M = df_feat.stat.sampleBy("stratum_id", fractions, seed=42).cache()
n_M = M.count()

print(f"|M| = {n_M:,} (objetivo {N_M_TARGET:,}, esperado ~5.03M)")
assert abs(n_M - ESTIMATED_M) / ESTIMATED_M < 0.02, f"|M| diverge: {n_M:,}"

M.select("stratum_id", "fare_amount", "trip_distance", "pu_macro_zone", "payment_group").show(5, truncate=False)


|M| = 5,029,725 (objetivo 5,000,000, esperado ~5.03M)
+--------------------------------------+-----------+-------------+-------------+-------------+
|stratum_id                            |fare_amount|trip_distance|pu_macro_zone|payment_group|
+--------------------------------------+-----------+-------------+-------------+-------------+
|manhattan|credit|late_night|medium    |22.6       |5.72         |manhattan    |credit       |
|manhattan|credit|late_night|medium    |35.9       |7.2          |manhattan    |credit       |
|manhattan|credit|late_night|medium    |16.3       |3.67         |manhattan    |credit       |
|outer_borough|credit|late_night|medium|14.2       |2.67         |outer_borough|credit       |
|manhattan|credit|other|short          |8.6        |0.87         |manhattan    |credit       |
+--------------------------------------+-----------+-------------+-------------+-------------+
only showing top 5 rows


### 2.1 Construcción de M' a partir de M

> **Nota al profesor:** esta sección reutiliza de forma idéntica la técnica de la Actividad 3: ventana exacta estratificada con piso determinístico de 50 filas por estrato y fracción global 0.20. Se mantiene para preservar continuidad metodológica con la muestra individual que fue validada en la entrega anterior.

A partir de M se construye M' como subconjunto individual manejable. Para cada estrato `s`, se define `target_n_s = min(n_M_s, max(50, floor(0.20 * n_M_s)))`. Luego se ordenan aleatoriamente las filas dentro de cada `stratum_id` con semilla fija y se conservan las primeras `target_n_s`. Esto conserva todos los estratos y reduce el riesgo de perder perfiles raros. Se usa ventana exacta porque garantiza el conteo determinístico por estrato; `sampleBy` es Bernoulli y no asegura el piso en cada ejecución.


In [5]:
# Construcción de M' por ventana exacta con piso determinístico de 50.
F_GLOBAL_MP = 0.20
MIN_FLOOR_MP = 50

counts_M = M.groupBy("stratum_id").count().withColumnRenamed("count", "n_M_s")
target_Mp = counts_M.withColumn(
    "target_n_s",
    F.least(
        F.col("n_M_s"),
        F.greatest(F.lit(MIN_FLOOR_MP).cast("long"),
                   F.floor(F.lit(F_GLOBAL_MP) * F.col("n_M_s")).cast("long"))))

w_Mp = Window.partitionBy("stratum_id").orderBy(F.rand(seed=42))
M_prime = (M.join(target_Mp, "stratum_id")
    .withColumn("rn", F.row_number().over(w_Mp))
    .filter(F.col("rn") <= F.col("target_n_s"))
    .drop("rn", "target_n_s", "n_M_s")
    .cache())

n_Mp = M_prime.count()
print(f"|M'| = {n_Mp:,} (esperado ~1.0M)")
assert 950_000 <= n_Mp <= 1_100_000, f"|M'| fuera de rango: {n_Mp:,}"


|M'| = 1,006,344 (esperado ~1.0M)


### 2.2 Validación de representatividad M' vs M

> **Nota al profesor:** esta validación replica la verificación compacta de la Actividad 3. Se revisan tamaño, cobertura de estratos, piso mínimo y marginales de las cuatro variables de caracterización que definen `stratum_id`.

El propósito es confirmar que M' preserva la estructura de M antes de modelar: mantiene los 240 estratos, respeta el piso mínimo por estrato y conserva las proporciones marginales de zona, pago, franja horaria y rango de distancia.


In [6]:
# Validación compacta M' vs M.
n_M_v, n_Mp_v = M.count(), M_prime.count()
strata_M = M.select("stratum_id").distinct().count()
strata_Mp = M_prime.select("stratum_id").distinct().count()
min_count_Mp = M_prime.groupBy("stratum_id").count().agg(F.min("count")).first()[0]

print(f"|M| = {n_M_v:,}  |M'| = {n_Mp_v:,}  ratio = {n_Mp_v / n_M_v:.4f}")
print(f"Estratos en M = {strata_M}, en M' = {strata_Mp} (debe ser 240)")
print(f"Piso mínimo por estrato en M' = {min_count_Mp} (debe ser >= 50)\n")
assert strata_Mp == strata_M == 240, "M' perdió estratos"
assert min_count_Mp >= 50, f"Piso violado: {min_count_Mp}"

for col_name in ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]:
    p_M = {r[col_name]: r["count"] / n_M_v for r in M.groupBy(col_name).count().collect()}
    p_Mp = {r[col_name]: r["count"] / n_Mp_v for r in M_prime.groupBy(col_name).count().collect()}
    max_diff_pp = max(abs(p_M.get(k, 0) - p_Mp.get(k, 0)) for k in set(p_M) | set(p_Mp)) * 100
    print(f"  {col_name}: max |p_M - p_M'| = {max_diff_pp:.4f} pp")
    assert max_diff_pp < 0.5, f"Marginal de {col_name} diverge: {max_diff_pp:.4f} pp"


|M| = 5,029,725  |M'| = 1,006,344  ratio = 0.2001
Estratos en M = 240, en M' = 240 (debe ser 240)
Piso mínimo por estrato en M' = 50 (debe ser >= 50)

  pu_macro_zone: max |p_M - p_M'| = 0.0369 pp
  payment_group: max |p_M - p_M'| = 0.0307 pp
  day_hour_bucket: max |p_M - p_M'| = 0.0079 pp
  trip_distance_bin: max |p_M - p_M'| = 0.0287 pp


## 3. Preparación del conjunto de entrenamiento y prueba

En aprendizaje no supervisado no existe una etiqueta objetivo que proteger durante la partición. Aun así, el split train/test es útil para revisar estabilidad: el modelo aprende la estructura en train y después se evalúa si el patrón se mantiene en test mediante silhouette, tamaños de cluster y perfiles agregados.

Se mantiene el split estratificado exacto de la Actividad 3 para minimizar sesgos en las variables de caracterización. La partición usa una ventana por `stratum_id`, orden aleatorio reproducible y corte `floor(0.8 * n_s)` para train; el resto queda en test. Se usa 80/20 porque deja suficiente test para evaluar estabilidad sin sacrificar estructura de entrenamiento y, combinado con el piso de 50 en M', garantiza al menos 10 filas por estrato en test. A diferencia de `randomSplit` o `sampleBy`, este método garantiza conteos determinísticos por estrato y conserva todos los perfiles raros protegidos por el piso de M'.


In [7]:
# Split estratificado exacto 80/20 sobre M_prime.
TRAIN_RATIO = 0.8

counts_Mp = M_prime.groupBy("stratum_id").count().withColumnRenamed("count", "n_Mp_s")
w_split = Window.partitionBy("stratum_id").orderBy(F.rand(seed=123))

Mp_with_rn = (M_prime.join(counts_Mp, "stratum_id")
    .withColumn("rn", F.row_number().over(w_split))
    .withColumn("train_cutoff", F.floor(F.lit(TRAIN_RATIO) * F.col("n_Mp_s")).cast("long")))

train_df = (Mp_with_rn.filter(F.col("rn") <= F.col("train_cutoff"))
            .drop("rn", "train_cutoff", "n_Mp_s").cache())
test_df  = (Mp_with_rn.filter(F.col("rn") >  F.col("train_cutoff"))
            .drop("rn", "train_cutoff", "n_Mp_s").cache())

n_train, n_test = train_df.count(), test_df.count()
print(f"|train| = {n_train:,}  |test| = {n_test:,}  ratio_train = {n_train / (n_train + n_test):.4f}")


|train| = 804,991  |test| = 201,353  ratio_train = 0.7999


In [8]:
# Verificación post-split: cobertura de estratos, piso y marginales train vs test.
strata_train = train_df.select("stratum_id").distinct().count()
strata_test = test_df.select("stratum_id").distinct().count()
min_train = train_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]
min_test = test_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]

print(f"Estratos train = {strata_train}, test = {strata_test} (esperado 240)")
print(f"Piso min en train = {min_train}, en test = {min_test} (test debe ser >= 10)\n")
assert strata_train == 240 and strata_test == 240
assert min_test >= 10, f"Piso en test violado: {min_test}"

for col_name in ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]:
    p_tr = {r[col_name]: r["count"] / n_train for r in train_df.groupBy(col_name).count().collect()}
    p_te = {r[col_name]: r["count"] / n_test for r in test_df.groupBy(col_name).count().collect()}
    max_diff_pp = max(abs(p_tr.get(k, 0) - p_te.get(k, 0)) for k in set(p_tr) | set(p_te)) * 100
    print(f"  {col_name}: max |p_train - p_test| = {max_diff_pp:.4f} pp")
    assert max_diff_pp < 0.5, f"Marginal de {col_name} diverge: {max_diff_pp:.4f} pp"


Estratos train = 240, test = 240 (esperado 240)
Piso min en train = 40, en test = 10 (test debe ser >= 10)

  pu_macro_zone: max |p_train - p_test| = 0.0315 pp
  payment_group: max |p_train - p_test| = 0.0215 pp
  day_hour_bucket: max |p_train - p_test| = 0.0066 pp
  trip_distance_bin: max |p_train - p_test| = 0.0176 pp


## 4. Construcción de modelos de aprendizaje no supervisado

### 4.1 Definición del problema y variables de agrupamiento

El problema no supervisado es **segmentar viajes Yellow Taxi en arquetipos operativos**. A diferencia de la Actividad 3, no se define una variable objetivo: el modelo busca estructura interna en los viajes a partir de variables de forma, contexto y régimen operativo.

El set base reutiliza las variables operativas de la Actividad 3, excluyendo `fare_amount`, `total_amount` y componentes de cobro. La exclusión no es por fuga de etiqueta, porque aquí no hay etiqueta, sino por enfoque: incluir montos produciría perfiles económico-tarifarios dominados por precio, no arquetipos operativos del viaje.

| Feature | Tipo | Justificación |
|---|---|---|
| `trip_distance` | numérica continua | Distancia recorrida; principal dimensión física del trayecto |
| `trip_duration_min` | numérica continua | Duración real del viaje; separa viajes largos por tiempo de viajes largos por distancia |
| `average_speed_mph_capped` | numérica continua | Velocidad promedio del viaje (`distancia / duración`), capada a 40 mph para evitar que outliers dominen KMeans |
| `passenger_count` | numérica entera | Información operativa básica de ocupación |
| `pu_macro_zone` | categórica | Resume el origen geográfico en aeropuerto, Manhattan, outer borough o unknown |
| `RatecodeID` | categórica | Distingue régimen tarifario operativo, incluidos códigos especiales y Flex imputado |
| `day_hour_bucket` | categórica | Captura contexto temporal: madrugada, fin de semana, picos laborales y resto |
| `is_flex_fare` | binaria | Identifica viajes con tarifa upfront del régimen Flex Fare |

`cbd_period_flag` se conserva para perfilar clusters, pero no entra al vector de entrenamiento. La primera corrida mostró que incluirla elevaba la silhouette, pero separaba dos grupos grandes principalmente por periodo regulatorio (`pre_cbd` vs `post_cbd`). Como el objetivo son arquetipos operativos de viaje, no una partición calendario/regulación, se excluye desde el diseño del modelo.

`average_speed_mph` no representa la velocidad instantánea del taxi, sino la velocidad promedio de todo el viaje: distancia total dividida entre duración total. Para entrenamiento se usa `average_speed_mph_capped`, limitada a 40 mph. Ese umbral queda por encima del percentil 99 observado y conserva más del 99% de la variación normal, pero comprime la cola extrema de velocidades promedio físicamente improbables.


### 4.2 Derivación y selección de features

Igual que en la Actividad 3, `trip_duration_min` se deriva después del split y por separado en train y test. Esto mantiene la sección 2 dedicada a la muestra y deja las variables de modelado dentro de la sección de modelos. El filtro de duración es determinista por fila, por lo que no introduce fuga de información: sólo elimina registros con duración imposible o incompleta en cada partición.


In [9]:
# Derivación de duración y variables auxiliares para clustering.
def add_trip_duration(df):
    return (df
        .withColumn("trip_duration_min_raw",
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / F.lit(60.0))
        .filter(F.col("trip_duration_min_raw").between(0.5, 720.0))
        .withColumn("trip_duration_min", F.col("trip_duration_min_raw").cast("float"))
        .drop("trip_duration_min_raw"))

AVERAGE_SPEED_CAP_MPH = 40.0

train_ml = (add_trip_duration(train_df)
    .withColumn("is_flex_fare", F.col("is_flex_fare").cast("byte"))
    .withColumn("average_speed_mph", F.col("trip_distance") / (F.col("trip_duration_min") / F.lit(60.0)))
    .withColumn("average_speed_mph_capped", F.least(F.col("average_speed_mph"), F.lit(AVERAGE_SPEED_CAP_MPH)))
    .cache())

test_ml = (add_trip_duration(test_df)
    .withColumn("is_flex_fare", F.col("is_flex_fare").cast("byte"))
    .withColumn("average_speed_mph", F.col("trip_distance") / (F.col("trip_duration_min") / F.lit(60.0)))
    .withColumn("average_speed_mph_capped", F.least(F.col("average_speed_mph"), F.lit(AVERAGE_SPEED_CAP_MPH)))
    .cache())

n_train_ml, n_test_ml = train_ml.count(), test_ml.count()
print(f"Filas antes de duración: train={n_train:,}  test={n_test:,}")
print(f"Filas tras filtro de duración [0.5, 720] min: train={n_train_ml:,}  test={n_test_ml:,}")
print(f"Pérdida train: {(n_train - n_train_ml) / n_train * 100:.3f}%")
print(f"Pérdida test : {(n_test - n_test_ml) / n_test * 100:.3f}%")


Filas antes de duración: train=804,991  test=201,353
Filas tras filtro de duración [0.5, 720] min: train=797,458  test=199,348
Pérdida train: 0.936%
Pérdida test : 0.996%


In [10]:
# Diagnóstico de average_speed_mph: correlaciones, percentiles y valores extremos.
corr_pairs = [
    ("trip_distance", "trip_duration_min"),
    ("trip_distance", "average_speed_mph"),
    ("trip_duration_min", "average_speed_mph"),
]

print("Correlaciones en train:")
for a, b in corr_pairs:
    r = train_ml.agg(F.corr(a, b).alias("r")).first()["r"]
    print(f"  corr({a}, {b}) = {r:.4f}")

quantile_probs = [0.0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1.0]
quantile_values = train_ml.approxQuantile("average_speed_mph", quantile_probs, 0.001)
print("\nPercentiles de average_speed_mph en train:")
for p, value in zip(quantile_probs, quantile_values):
    label = "max" if p == 1.0 else f"p{int(p * 1000) / 10:g}"
    print(f"  {label:>5}: {value:8.2f} mph")

speed_thresholds = [40, 60, 80, 100]
print("\nViajes por encima de umbrales de velocidad promedio:")
for threshold in speed_thresholds:
    n_threshold = train_ml.filter(F.col("average_speed_mph") > threshold).count()
    pct_threshold = n_threshold / n_train_ml * 100
    print(f"  average_speed_mph > {threshold:>3}: {n_threshold:>6,} ({pct_threshold:.4f}%)")


Correlaciones en train:
  corr(trip_distance, trip_duration_min) = 0.7416
  corr(trip_distance, average_speed_mph) = 0.3502
  corr(trip_duration_min, average_speed_mph) = 0.0928

Percentiles de average_speed_mph en train:
     p0:     0.00 mph
    p25:     6.94 mph
    p50:     9.38 mph
    p75:    12.96 mph
    p90:    19.10 mph
    p95:    24.24 mph
    p99:    34.65 mph
  p99.9:  4482.86 mph
    max:  4482.86 mph

Viajes por encima de umbrales de velocidad promedio:
  average_speed_mph >  40:  2,973 (0.3728%)
  average_speed_mph >  60:    182 (0.0228%)
  average_speed_mph >  80:    134 (0.0168%)
  average_speed_mph > 100:    112 (0.0140%)


**Decisión sobre `average_speed_mph`.**

`average_speed_mph` es la velocidad promedio del viaje completo, no una medición instantánea. Aporta un eje distinto respecto a duración y distancia, pero su versión cruda presenta una cola extrema. Para evitar que KMeans quede dominado por esos valores, se usa `average_speed_mph_capped <= 40` como feature final. En corridas exploratorias, el modelo sin `cbd_period_flag` pasó de silhouette test 0.2966 sin velocidad capada a 0.4176 con velocidad capada, y los perfiles resultantes fueron operativamente interpretables. Por simplicidad, las variantes exploratorias no se conservan como celdas de modelado.


In [11]:
# Set final de features de entrenamiento; cbd_period_flag y average_speed_mph cruda quedan sólo para análisis/perfilado.
BASE_NUM_COLS = ["trip_distance", "trip_duration_min", "passenger_count", "average_speed_mph_capped"]
NUM_COLS = BASE_NUM_COLS

CAT_COLS = ["pu_macro_zone", "RatecodeID", "day_hour_bucket"]
BIN_COLS = ["is_flex_fare"]
PROFILE_ONLY_COLS = ["cbd_period_flag", "average_speed_mph"]
FEATURE_COLS = NUM_COLS + CAT_COLS + BIN_COLS
EXCLUDED_AMOUNT_COLS = {
    "fare_amount", "total_amount", "tip_amount", "tolls_amount", "extra", "mta_tax",
    "improvement_surcharge", "congestion_surcharge", "Airport_fee", "cbd_congestion_fee",
}

assert not (set(FEATURE_COLS) & EXCLUDED_AMOUNT_COLS), "Hay columnas de monto en FEATURE_COLS"
assert "cbd_period_flag" not in FEATURE_COLS, "cbd_period_flag sólo debe usarse para perfilado"
assert "average_speed_mph" not in FEATURE_COLS, "average_speed_mph cruda sólo debe usarse para perfilado"
missing = sorted(set(FEATURE_COLS + PROFILE_ONLY_COLS) - set(train_ml.columns))
assert not missing, f"Columnas ausentes en train_ml: {missing}"

print(f"Features de entrenamiento ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Sólo perfilado: {PROFILE_ONLY_COLS}")


Features de entrenamiento (8): ['trip_distance', 'trip_duration_min', 'passenger_count', 'average_speed_mph_capped', 'pu_macro_zone', 'RatecodeID', 'day_hour_bucket', 'is_flex_fare']
Sólo perfilado: ['cbd_period_flag', 'average_speed_mph']


### 4.3 Pipeline de preprocesamiento

El pipeline separa las variables continuas de las categóricas. Las continuas se ensamblan y se escalan con `StandardScaler`, porque KMeans y GMM dependen de distancias. Las categóricas se codifican con `StringIndexer` y `OneHotEncoder` para evitar que Spark interprete categorías como números ordinales. La binaria `is_flex_fare` se conserva como 0/1.

`cbd_period_flag` y `average_speed_mph` cruda no entran al vector de features; quedan disponibles para interpretación posterior. Los transformadores que aprenden parámetros (`StringIndexer`, `OneHotEncoder`, `StandardScaler`) se ajustan sólo con train y luego transforman test mediante el `Pipeline`, evitando que estadísticas de test influyan en la preparación del modelo.


In [12]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans, GaussianMixture
from pyspark.ml.evaluation import ClusteringEvaluator

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CAT_COLS]
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in CAT_COLS],
    outputCols=[f"{c}_ohe" for c in CAT_COLS],
)
num_assembler = VectorAssembler(inputCols=NUM_COLS, outputCol="num_features", handleInvalid="keep")
scaler = StandardScaler(inputCol="num_features", outputCol="scaled_num_features", withMean=True, withStd=True)
feature_assembler = VectorAssembler(
    inputCols=["scaled_num_features"] + [f"{c}_ohe" for c in CAT_COLS] + BIN_COLS,
    outputCol="features",
    handleInvalid="keep",
)

preprocessing_stages = indexers + [encoder, num_assembler, scaler, feature_assembler]
print(f"Etapas de preprocesamiento: {len(preprocessing_stages)}")
print(f"Continuas escaladas: {NUM_COLS}")
print(f"Categóricas OHE: {CAT_COLS}")
print(f"Binarias sin escalar: {BIN_COLS}")


Etapas de preprocesamiento: 7
Continuas escaladas: ['trip_distance', 'trip_duration_min', 'passenger_count', 'average_speed_mph_capped']
Categóricas OHE: ['pu_macro_zone', 'RatecodeID', 'day_hour_bucket']
Binarias sin escalar: ['is_flex_fare']


### 4.4 Selección de k con KMeans

KMeans agrupa observaciones minimizando la distancia cuadrática al centroide de cada cluster. Como usa distancias, el escalado definido en el pipeline es obligatorio para que `trip_distance`, `trip_duration_min`, `passenger_count` y `average_speed_mph_capped` tengan una contribución comparable.

La selección de `k` se hace con un barrido manual de 2 a 7 clusters sobre el set final de features. La métrica principal es silhouette con distancia euclidiana cuadrática, complementada con estabilidad train/test y balance de tamaños. No se usa `ParamGridBuilder` ni validación cruzada porque no hay etiqueta supervisada: elegir `k` sólo por el máximo automático de una métrica interna puede producir una solución poco interpretable.


In [13]:
# Ajuste del preprocesamiento sólo con train; test sólo se transforma.
preprocess_pipeline = Pipeline(stages=preprocessing_stages)
preprocess_model = preprocess_pipeline.fit(train_ml)

train_features = preprocess_model.transform(train_ml).cache()
test_features = preprocess_model.transform(test_ml).cache()

n_train_features = train_features.count()
n_test_features = test_features.count()

print(f"Filas con features: train={n_train_features:,}  test={n_test_features:,}")
print("Vector final: features")


Filas con features: train=797,458  test=199,348
Vector final: features


In [14]:
# Barrido manual de k para KMeans.
K_VALUES = list(range(2, 8))
KMEANS_SEED = 42
KMEANS_MAX_ITER = 30

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

kmeans_models = {}
kmeans_results = []

for k in K_VALUES:
    print(f"Entrenando KMeans con k={k}...")
    kmeans = KMeans(
        featuresCol="features",
        predictionCol="cluster",
        k=k,
        seed=KMEANS_SEED,
        maxIter=KMEANS_MAX_ITER,
    )
    model = kmeans.fit(train_features)
    kmeans_models[k] = model

    pred_train = model.transform(train_features).select("features", "cluster")
    pred_test = model.transform(test_features).select("features", "cluster")

    sil_train = evaluator.evaluate(pred_train)
    sil_test = evaluator.evaluate(pred_test)

    cluster_sizes = pred_train.groupBy("cluster").count().cache()
    size_stats = cluster_sizes.agg(
        F.min("count").alias("min_cluster"),
        F.max("count").alias("max_cluster"),
    ).first()
    cluster_sizes.unpersist()

    max_cluster_fraction = size_stats["max_cluster"] / n_train_features
    kmeans_results.append({
        "k": k,
        "silhouette_train": round(float(sil_train), 4),
        "silhouette_test": round(float(sil_test), 4),
        "min_cluster": int(size_stats["min_cluster"]),
        "max_cluster": int(size_stats["max_cluster"]),
        "max_cluster_fraction": round(float(max_cluster_fraction), 4),
    })

results_df = spark.createDataFrame(kmeans_results).orderBy("k")
results_df.show(truncate=False)


Entrenando KMeans con k=2...


26/06/01 01:24:33 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Entrenando KMeans con k=3...
Entrenando KMeans con k=4...
Entrenando KMeans con k=5...
Entrenando KMeans con k=6...
Entrenando KMeans con k=7...
+---+-----------+--------------------+-----------+---------------+----------------+
|k  |max_cluster|max_cluster_fraction|min_cluster|silhouette_test|silhouette_train|
+---+-----------+--------------------+-----------+---------------+----------------+
|2  |684832     |0.8588              |112626     |0.6845         |0.6837          |
|3  |570361     |0.7152              |107447     |0.492          |0.4908          |
|4  |524758     |0.658               |47962      |0.4696         |0.4694          |
|5  |450348     |0.5647              |44261      |0.4176         |0.4177          |
|6  |288705     |0.362               |40375      |0.235          |0.2352          |
|7  |324487     |0.4069              |25200      |0.2735         |0.2732          |
+---+-----------+--------------------+-----------+---------------+----------------+



### 4.5 KMeans final e interpretación

El barrido de `k` no se decide sólo por la silhouette. `k=2` y `k=3` obtienen métricas altas, pero concentran demasiados viajes en el cluster dominante (`85.88%` y `71.52%`, respectivamente). `k=4` mantiene una silhouette mayor que `k=5`, pero todavía deja `65.80%` de los viajes en un solo cluster, por lo que la segmentación sigue siendo muy gruesa.

Se usa `k=5` porque reduce el cluster dominante a `56.47%`, mantiene estabilidad train/test y abre espacio para una segmentación más granular sin caer en la pérdida de calidad que aparece con `k=6` y `k=7`. Los nombres de los arquetipos no se fijan en este punto: se asignan después, a partir del perfil operativo de los clusters.


In [15]:
# KMeans final con el set de features seleccionado.
SELECTED_K = 5

kmeans_final = kmeans_models.get(SELECTED_K)
if kmeans_final is None:
    kmeans_final = KMeans(
        featuresCol="features",
        predictionCol="cluster",
        k=SELECTED_K,
        seed=KMEANS_SEED,
        maxIter=KMEANS_MAX_ITER,
    ).fit(train_features)

kmeans_train = kmeans_final.transform(train_features).cache()
kmeans_test = kmeans_final.transform(test_features).cache()

final_sil_train = evaluator.evaluate(kmeans_train.select("features", "cluster"))
final_sil_test = evaluator.evaluate(kmeans_test.select("features", "cluster"))

print(f"KMeans final k={SELECTED_K}")
print(f"Silhouette train = {final_sil_train:.4f}")
print(f"Silhouette test  = {final_sil_test:.4f}")

final_cluster_counts = (kmeans_train.groupBy("cluster").count()
    .withColumn("cluster_fraction", F.round(F.col("count") / F.lit(n_train_features), 4))
    .orderBy("cluster"))
final_cluster_counts.show(truncate=False)


KMeans final k=5
Silhouette train = 0.4177
Silhouette test  = 0.4176
+-------+------+----------------+
|cluster|count |cluster_fraction|
+-------+------+----------------+
|0      |450348|0.5647          |
|1      |110819|0.139           |
|2      |74066 |0.0929          |
|3      |44261 |0.0555          |
|4      |117964|0.1479          |
+-------+------+----------------+



In [16]:
# Perfil operativo de clusters KMeans finales.
def top_category_by_cluster(df, col_name, cluster_col="cluster"):
    w_top = Window.partitionBy(cluster_col).orderBy(F.desc("count"), F.asc(col_name))
    return (df.groupBy(cluster_col, col_name).count()
        .withColumn("rn", F.row_number().over(w_top))
        .filter(F.col("rn") == 1)
        .select(
            F.col(cluster_col),
            F.col(col_name).alias(f"top_{col_name}"),
            F.col("count").alias(f"top_{col_name}_n"),
        ))

numeric_profile = (kmeans_train.groupBy("cluster")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
        F.round(F.expr("percentile_approx(trip_distance, 0.5, 1000)"), 2).alias("median_distance_mi"),
        F.round(F.avg("trip_duration_min"), 2).alias("avg_duration_min"),
        F.round(F.expr("percentile_approx(trip_duration_min, 0.5, 1000)"), 2).alias("median_duration_min"),
        F.round(F.avg("average_speed_mph"), 2).alias("mean_average_speed_mph"),
        F.round(F.expr("percentile_approx(average_speed_mph, 0.5, 1000)"), 2).alias("median_average_speed_mph"),
        F.round(F.avg("average_speed_mph_capped"), 2).alias("mean_average_speed_capped_mph"),
        F.round(F.expr("percentile_approx(average_speed_mph_capped, 0.5, 1000)"), 2).alias("median_average_speed_capped_mph"),
        F.round(F.avg("passenger_count"), 2).alias("avg_passengers"),
        F.round(F.avg("is_flex_fare"), 4).alias("fraction_flex"),
    ))

profile = numeric_profile
for col_name in ["pu_macro_zone", "RatecodeID", "day_hour_bucket", "cbd_period_flag"]:
    profile = profile.join(top_category_by_cluster(kmeans_train, col_name), "cluster", "left")

profile.orderBy("cluster").show(truncate=False)


+-------+------+---------------+------------------+----------------+-------------------+----------------------+------------------------+-----------------------------+-------------------------------+--------------+-------------+-----------------+-------------------+--------------+----------------+-------------------+---------------------+-------------------+---------------------+
|cluster|n     |avg_distance_mi|median_distance_mi|avg_duration_min|median_duration_min|mean_average_speed_mph|median_average_speed_mph|mean_average_speed_capped_mph|median_average_speed_capped_mph|avg_passengers|fraction_flex|top_pu_macro_zone|top_pu_macro_zone_n|top_RatecodeID|top_RatecodeID_n|top_day_hour_bucket|top_day_hour_bucket_n|top_cbd_period_flag|top_cbd_period_flag_n|
+-------+------+---------------+------------------+----------------+-------------------+----------------------+------------------------+-----------------------------+-------------------------------+--------------+-------------+---------

### 4.6 GaussianMixture como comparación probabilística

GaussianMixture Model se entrena con el mismo vector final de features que KMeans. Esto permite comparar dos formas de agrupamiento sobre la misma representación: KMeans asigna cada viaje al centroide más cercano, mientras que GMM calcula probabilidades de pertenencia a componentes.


In [17]:
# GMM con el mismo k y el mismo set final de features.
GMM_MAX_ITER = 20

gmm = GaussianMixture(
    featuresCol="features",
    predictionCol="gmm_cluster",
    probabilityCol="gmm_probability",
    k=SELECTED_K,
    seed=KMEANS_SEED,
    maxIter=GMM_MAX_ITER,
)

gmm_model = gmm.fit(train_features)

gmm_train = gmm_model.transform(train_features).cache()
gmm_test = gmm_model.transform(test_features).cache()

gmm_evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="gmm_cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

gmm_sil_train = gmm_evaluator.evaluate(gmm_train.select("features", "gmm_cluster"))
gmm_sil_test = gmm_evaluator.evaluate(gmm_test.select("features", "gmm_cluster"))

from pyspark.ml.functions import vector_to_array

gmm_train_profile = gmm_train.withColumn("gmm_prob_max", F.array_max(vector_to_array("gmm_probability")))

gmm_counts = (gmm_train_profile.groupBy("gmm_cluster")
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / F.lit(n_train_features), 4).alias("cluster_fraction"),
        F.round(F.avg("gmm_prob_max"), 4).alias("avg_max_probability"),
    )
    .orderBy("gmm_cluster"))

print(f"GMM k={SELECTED_K}")
print(f"Silhouette train = {gmm_sil_train:.4f}")
print(f"Silhouette test  = {gmm_sil_test:.4f}")
print(f"Log-likelihood train = {gmm_model.summary.logLikelihood:.2f}")
gmm_counts.show(truncate=False)


26/06/01 01:25:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


GMM k=5
Silhouette train = 0.3338
Silhouette test  = 0.3332
Log-likelihood train = 17798446.68
+-----------+------+----------------+-------------------+
|gmm_cluster|count |cluster_fraction|avg_max_probability|
+-----------+------+----------------+-------------------+
|0          |37145 |0.0466          |0.9957             |
|1          |189327|0.2374          |0.9991             |
|2          |529263|0.6637          |0.9993             |
|3          |32326 |0.0405          |0.9998             |
|4          |9397  |0.0118          |0.9244             |
+-----------+------+----------------+-------------------+



### 4.7 Comparación de modelos

KMeans supera a GaussianMixture en esta representación: KMeans obtiene silhouette test `0.4176`, mientras que GMM obtiene `0.3332`. Además, GMM concentra cerca de dos tercios de los viajes en un solo componente, lo que reduce su utilidad como segmentación operativa.

La comparación se interpreta con cautela porque `ClusteringEvaluator` usa `squaredEuclidean` por defecto, una geometría alineada con KMeans. Por eso la decisión no se basa sólo en la métrica: también se revisan estabilidad train/test, balance de clusters e interpretación de los perfiles. Con esos criterios, KMeans es el modelo final más adecuado para esta actividad.


### 4.8 Interpretación final

El modelo principal es **KMeans con k=5** sobre el vector final de 8 features. La silhouette es estable entre train y test (`0.4177` vs `0.4176`), lo que indica que la estructura aprendida en train también organiza de forma consistente los viajes no usados para ajustar el modelo.

Los cinco arquetipos operativos son:

| Cluster | Nombre operativo | Lectura |
|---|---|---|
| 0 | Urbano regular Manhattan | Viajes cortos, baja velocidad promedio, un pasajero, sin Flex, origen principalmente Manhattan |
| 1 | Flex urbano | Viajes urbanos medianos con `RatecodeID=99` y alta proporción Flex |
| 2 | Medianos rápidos | Viajes de mayor distancia y velocidad promedio, principalmente regulares |
| 3 | Aeropuerto/largos | Viajes largos, mayor duración, origen aeropuerto y presencia de tarifa especial |
| 4 | Viajes grupales Manhattan | Viajes cortos/medios con mayor número de pasajeros y sin Flex |

`RatecodeID=99` corresponde al grupo imputado para valores nulos de `RatecodeID`; en esta muestra queda fuertemente asociado con viajes Flex/upfront, por eso el cluster 1 se interpreta como Flex urbano y no como un código tarifario oficial adicional.

La variable `cbd_period_flag` se conserva sólo para perfilado. Esto permite observar diferencias temporales sin dejar que el modelo se convierta en una partición `pre_cbd` vs `post_cbd`. La velocidad usada en entrenamiento es `average_speed_mph_capped`, no la velocidad cruda: el cap en 40 mph queda por encima del percentil 99 y evita que outliers extremos dominen la distancia.


## Conclusiones

El aprendizaje no supervisado permitió transformar la muestra individual M' en una segmentación operativa de viajes Yellow Taxi sin utilizar una etiqueta objetivo. En este contexto, un **arquetipo** es un perfil típico de viaje descubierto a partir de similitudes en distancia, duración, velocidad promedio, zona, horario, régimen operativo y pasajeros; no es una etiqueta verdadera, sino una síntesis interpretable de un patrón recurrente.

La solución final usa KMeans con `k=5` y un vector de features operativo. Se excluyen montos para evitar perfiles económico-tarifarios y se excluye `cbd_period_flag` del entrenamiento para evitar una segmentación puramente regulatoria. `average_speed_mph_capped` se incluye porque aporta separación adicional sin permitir que velocidades promedio extremas distorsionen los centroides.

La utilidad de estos arquetipos es convertir millones de viajes individuales en grupos accionables. Los perfiles permiten distinguir viajes urbanos regulares, Flex urbano, trayectos medianos rápidos, viajes largos/aeropuerto y viajes grupales. Esa lectura puede apoyar planeación de flota, análisis de demanda por tipo de servicio, monitoreo de cambios operativos, revisión de viajes atípicos y comunicación ejecutiva más clara que una tabla de millones de registros.

Limitaciones: los clusters dependen de la escala, del valor de `k`, del cap elegido para velocidad promedio y de la geometría euclidiana de KMeans. No existe etiqueta externa para medir exactitud; por ello, la evaluación combina silhouette, estabilidad train/test, balance de tamaños e interpretación operativa. Como trabajo futuro, se podría probar PCA antes de clustering, validación temporal por mes o una comparación con BisectingKMeans.

**Declaración de uso de IA**

Google. (2026). *Gemini 3.5 Flash* [Modelo de lenguaje grande], utilizado para el proceso de aprendizaje del contenido de la semana y la validación de errores conceptuales y de código. https://deepmind.google/models/gemini/flash/
